# Chi-square tests
Goodness of fit and the test of independence, by hand and with scipy, then the Titanic case study.
The 60-person table comes from `570-choosing-a-hypothesis-test/data/people.csv`; the counts are typed in here.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

## 1. Goodness of fit: age groups against census shares 25% / 55% / 20%

In [2]:
observed = np.array([20, 26, 14])          # child, adult, elderly
shares = np.array([0.25, 0.55, 0.20])
expected = observed.sum() * shares          # n times each claimed share
parts = (observed - expected) ** 2 / expected
print('expected', expected, ' parts', parts.round(3))
chi2 = parts.sum()
df = len(observed) - 1
print(f'chi2 = {chi2:.3f}, df = {df}, p = {stats.chi2.sf(chi2, df):.3f}')
print(stats.chisquare(observed, expected))

expected [15. 33. 12.]  parts [1.667 1.485 0.333]
chi2 = 3.485, df = 2, p = 0.175
Power_divergenceResult(statistic=np.float64(3.484848484848485), pvalue=np.float64(0.17509541196140194))


## 2. Two categories: chi-square equals z squared

In [3]:
gof = stats.chisquare([26, 34], [30, 30])
z = (26 / 60 - 0.5) / np.sqrt(0.25 / 60)
print('chi2 =', round(gof.statistic, 4), ' z^2 =', round(z ** 2, 4), ' p =', round(gof.pvalue, 4))

chi2 = 1.0667  z^2 = 1.0667  p = 0.3017


## 3. Critical values of the chi-square distribution

In [4]:
for df in [1, 2, 3]:
    print(df, 'df: 5% critical value', round(stats.chi2.ppf(0.95, df), 2))

1 df: 5% critical value 3.84
2 df: 5% critical value 5.99
3 df: 5% critical value 7.81


## 4. Test of independence by hand: gender by age group

In [5]:
table = pd.DataFrame([[12, 12, 10], [8, 14, 4]], index=['female', 'male'],
                     columns=['child', 'adult', 'elderly'])
n = table.values.sum()
# expected = row total * column total / n, for every cell at once
expected = np.outer(table.sum(axis=1), table.sum(axis=0)) / n
print(expected.round(2))
parts = (table.values - expected) ** 2 / expected
print(parts.round(3))
chi2 = parts.sum()
df = (table.shape[0] - 1) * (table.shape[1] - 1)
print(f'chi2 = {chi2:.2f}, df = {df}, p = {stats.chi2.sf(chi2, df):.2f}')

[[11.33 14.73  7.93]
 [ 8.67 11.27  6.07]]
[[0.039 0.507 0.538]
 [0.051 0.663 0.704]]
chi2 = 2.50, df = 2, p = 0.29


In [6]:
result = stats.chi2_contingency(table)
print(result.statistic, result.pvalue, result.dof)

2.503107751976529 0.2860599496440383 2


## 5. Titanic: sex and survival

In [7]:
titanic = pd.read_csv('data/titanic_train.csv')
sex = pd.crosstab(titanic['Sex'], titanic['Survived'])
print(sex)
plain = stats.chi2_contingency(sex, correction=False)
print('expected:\n', plain.expected_freq.round(1))
print('plain chi2', round(plain.statistic, 1), ' p', plain.pvalue)
yates = stats.chi2_contingency(sex)       # Yates' correction is the default for 2 x 2
print('Yates chi2', round(yates.statistic, 1))
print("Cramer's V", round(np.sqrt(plain.statistic / len(titanic)), 2))

Survived    0    1
Sex               
female     81  233
male      468  109
expected:
 [[193.5 120.5]
 [355.5 221.5]]
plain chi2 263.1  p 3.711747770113424e-59


Yates chi2 260.7
Cramer's V 0.54


## 6. Titanic: class and survival

In [8]:
cls = pd.crosstab(titanic['Pclass'], titanic['Survived'])
res = stats.chi2_contingency(cls)
print(cls)
print('expected:\n', res.expected_freq.round(1))
print(f'chi2 = {res.statistic:.1f}, df = {res.dof}, p = {res.pvalue:.1e}')
print('survival rate by class', (cls[1] / cls.sum(axis=1)).round(2).tolist())
print("Cramer's V", round(stats.contingency.association(cls, method='cramer'), 2))

Survived    0    1
Pclass            
1          80  136
2          97   87
3         372  119
expected:
 [[133.1  82.9]
 [113.4  70.6]
 [302.5 188.5]]
chi2 = 102.9, df = 2, p = 4.5e-23
survival rate by class [0.63, 0.47, 0.24]
Cramer's V 0.34


## 7. Testing the chi-square approximation by simulation
Draw many samples with $H_0$ true and look at the $\chi^2$ values: their mean should be the df, and 5% of them should pass the 5% critical value.

In [9]:
rng = np.random.default_rng(0)
crit = stats.chi2.ppf(0.95, 2)
def reject_rate(shares, n, reps=200_000):
    """Share of samples drawn with H0 true whose chi2 passes the 5% critical value."""
    shares = np.asarray(shares)
    counts = rng.multinomial(n, shares, size=reps)
    expected = n * shares
    chi = ((counts - expected) ** 2 / expected).sum(axis=1)
    return chi, (chi > crit).mean()
# census shares, n = 60: every expected count is 12 or more
chi, rate = reject_rate([0.25, 0.55, 0.20], 60)
print(f'shares 0.25/0.55/0.20, n = 60: smallest E = 12.0  mean chi2 = {chi.mean():.2f}  rejected = {rate:.3f}')
# two rare categories, tiny samples: expected counts far below 5
for n in [8, 10]:
    _, rate = reject_rate([0.90, 0.05, 0.05], n)
    print(f'shares 0.90/0.05/0.05, n = {n}: smallest E = {n * 0.05:.1f}  rejected = {rate:.3f}')

# Poisson goodness of fit with lambda estimated from the data: does df drop by one?
lam, n, reps = 2.0, 200, 5_000
chis = []
for _ in range(reps):
    y = rng.poisson(lam, size=n)
    lam_hat = y.mean()
    obs = np.array([(y == 0).sum(), (y == 1).sum(), (y == 2).sum(), (y == 3).sum(), (y >= 4).sum()])
    p = stats.poisson.pmf([0, 1, 2, 3], lam_hat)
    exp = n * np.append(p, 1 - p.sum())
    chis.append(((obs - exp) ** 2 / exp).sum())
chis = np.array(chis)
print(f'Poisson fit, 5 categories, lambda estimated: mean chi2 = {chis.mean():.2f}')
print(f'share above the df=4 critical value: {(chis > stats.chi2.ppf(0.95, 4)).mean():.3f}, '
      f'above the df=3 critical value: {(chis > stats.chi2.ppf(0.95, 3)).mean():.3f}')

shares 0.25/0.55/0.20, n = 60: smallest E = 12.0  mean chi2 = 2.00  rejected = 0.049
shares 0.90/0.05/0.05, n = 8: smallest E = 0.4  rejected = 0.112


shares 0.90/0.05/0.05, n = 10: smallest E = 0.5  rejected = 0.027


Poisson fit, 5 categories, lambda estimated: mean chi2 = 3.03
share above the df=4 critical value: 0.022, above the df=3 critical value: 0.051


## 8. scikit-learn's chi2 score on one-hot columns

In [10]:
from sklearn.feature_selection import chi2
onehot = pd.get_dummies(titanic['Sex']).astype(int)
scores, _ = chi2(onehot, titanic['Survived'])
print(dict(zip(onehot.columns, scores.round(1))), ' sum =', scores.sum().round(1))
print('test of independence, no correction:',
      round(stats.chi2_contingency(sex, correction=False).statistic, 1))

{'female': np.float64(170.3), 'male': np.float64(92.7)}  sum = 263.1
test of independence, no correction: 263.1
